In [1]:
import pandas as pd
import numpy as np

from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
df = pd.read_csv("../data/marketing_AB_clean.csv")

print(f"Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully.
Rows: 588,101
Columns: 6


In [3]:
print("Experiment groups:")
print(df["test group"].value_counts())

print("\nUnique users:")
print(df["user id"].nunique())

print("\nTotal observations:")
print(len(df))

Experiment groups:
test group
ad     564577
psa     23524
Name: count, dtype: int64

Unique users:
588101

Total observations:
588101


In [4]:
groups = sorted(df["test group"].unique())

print("Observed experiment groups:", groups)
print("Number of groups:", len(groups))

assert set(groups) == {"ad", "psa"}, "Unexpected experiment groups detected."

print("Treatment/control integrity check passed.")

Observed experiment groups: ['ad', 'psa']
Number of groups: 2
Treatment/control integrity check passed.


In [5]:
observed_counts = df["test group"].value_counts()

expected_ratio = {
    "ad": 0.96,
    "psa": 0.04
}

total_users = len(df)

expected_counts = pd.Series({
    group: total_users * expected_ratio[group]
    for group in expected_ratio
})

srm_table = pd.DataFrame({
    "observed": observed_counts,
    "expected": expected_counts
})

srm_table["difference"] = (
    srm_table["observed"] - srm_table["expected"]
)

srm_table["observed_pct"] = (
    srm_table["observed"] / total_users * 100
).round(4)

srm_table["expected_pct"] = (
    srm_table["expected"] / total_users * 100
).round(4)

srm_table

,observed,expected,difference,observed_pct,expected_pct
ad,564577,564576.96,0.04,96.0,96.0
psa,23524,23524.04,-0.04,4.0,4.0


In [6]:
observed = srm_table["observed"].values
expected = srm_table["expected"].values

chi2_stat, p_value = stats.chisquare(
    f_obs=observed,
    f_exp=expected
)

print(f"Chi-square statistic: {chi2_stat:.6f}")
print(f"p-value: {p_value:.6f}")

Chi-square statistic: 0.000000
p-value: 0.999788


In [7]:
alpha = 0.05

if p_value < alpha:
    print("SRM detected: reject the null hypothesis.")
else:
    print("No statistically significant SRM detected: fail to reject the null hypothesis.")

No statistically significant SRM detected: fail to reject the null hypothesis.


In [8]:
total_rows = len(df)
unique_users = df["user id"].nunique()
duplicate_rows = df["user id"].duplicated().sum()

print(f"Total observations : {total_rows:,}")
print(f"Unique users       : {unique_users:,}")
print(f"Duplicate rows     : {duplicate_rows:,}")

if total_rows == unique_users:
    print("\nUser-level uniqueness check passed.")
else:
    print("\nPotential repeated-user issue detected.")

Total observations : 588,101
Unique users       : 588,101
Duplicate rows     : 0

User-level uniqueness check passed.


In [9]:
group_summary = (
    df.groupby("test group")
    .agg(
        users=("user id", "count"),
        unique_users=("user id", "nunique")
    )
)

group_summary["duplicate_users"] = (
    group_summary["users"] - group_summary["unique_users"]
)

group_summary

,users,unique_users,duplicate_users
test group,,,
ad,564577,564577,0
psa,23524,23524,0


In [10]:
conversion_values = df["converted"].value_counts(dropna=False)

print("Conversion values:")
print(conversion_values)

print("\nData type:", df["converted"].dtype)
print("Missing values:", df["converted"].isna().sum())

Conversion values:
converted
False    573258
True      14843
Name: count, dtype: int64

Data type: bool
Missing values: 0


In [11]:
conversion_crosstab = pd.crosstab(
    df["test group"],
    df["converted"]
)

conversion_crosstab

converted,False,True
test group,,
ad,550154,14423
psa,23104,420


In [12]:
exposure_summary = (
    df.groupby("test group")["total ads"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75),
        max="max"
    )
    .round(2)
)

exposure_summary

,count,mean,median,q1,q3,max
test group,,,,,,
ad,564577,24.82,13.0,4.0,27.0,2065
psa,23524,24.76,12.0,4.0,26.0,907


In [13]:
for group in ["ad", "psa"]:
    group_data = df.loc[df["test group"] == group, "total ads"]
    
    print(f"\n{group.upper()}")
    print(f"> 100 ads : {(group_data > 100).sum():,}")
    print(f"> 500 ads : {(group_data > 500).sum():,}")
    print(f"> 1000 ads: {(group_data > 1000).sum():,}")


AD
> 100 ads : 22,054
> 500 ads : 566
> 1000 ads: 36

PSA
> 100 ads : 1,010
> 500 ads : 19
> 1000 ads: 0


In [14]:
exposure_tail = []

for group in ["ad", "psa"]:
    group_data = df.loc[df["test group"] == group, "total ads"]
    n = len(group_data)
    
    exposure_tail.append({
        "test_group": group,
        "users": n,
        "over_100_ads_pct": (group_data > 100).mean() * 100,
        "over_500_ads_pct": (group_data > 500).mean() * 100,
        "over_1000_ads_pct": (group_data > 1000).mean() * 100
    })

exposure_tail = pd.DataFrame(exposure_tail)

exposure_tail.round(4)

,test_group,users,over_100_ads_pct,over_500_ads_pct,over_1000_ads_pct
0,ad,564577,3.9063,0.1003,0.0064
1,psa,23524,4.2935,0.0808,0.0000


## Experiment Validation Verdict

### Sample Ratio Mismatch

The observed treatment allocation was 96% Ad and 4% PSA. Relative to the assumed 96/4 allocation, the SRM chi-square goodness-of-fit test produced a chi-square statistic of 0.000 and a p-value of 0.9998.

No statistically significant Sample Ratio Mismatch was detected.

However, the original intended allocation cannot be independently verified from the available dataset. Therefore, the SRM result should be interpreted as consistency with the assumed allocation rather than proof that randomization was correctly implemented.

### User-Level Independence

All 588,101 observations correspond to unique users. No duplicate `user_id` values were detected, so there is no obvious repeated-user issue in the dataset.

### Treatment and Outcome Integrity

The dataset contains exactly two experiment groups (`ad` and `psa`). The conversion outcome is binary Boolean with no missing values or unexpected values.

### Exposure Sanity Check

`total_ads` is strongly right-skewed, but no invalid exposure values were identified. Exposure-tail proportions were broadly similar between the Ad and PSA groups. Extreme exposure observations were retained because there was insufficient evidence to classify them as erroneous.

### Overall Assessment

The dataset passes the available structural and integrity checks required for the subsequent statistical analysis.

The experiment should not be described as fully validated because the original randomization mechanism and intended allocation cannot be independently verified from the dataset. Nevertheless, no major data-integrity issue was identified that would prevent proceeding with the primary conversion analysis.